# Memory Environment — Evaluation Notebook

Run baseline and RL agents across all scenarios and compare performance.

In [1]:
import sys
from pathlib import Path

# ensure project root is on sys.path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
from rich.console import Console
from rich.table import Table

from client import MemoryEnv, BaselineAgent, RLAgent, run_scenario
from server.services.action_handler import ACTION_LIST

console = Console()

## Metrics helper

In [2]:
def compute_metrics(episode_rewards, episode_infos):
    rewards = np.array(episode_rewards)
    out = {
        "mean_reward": float(np.mean(rewards)),
        "std_reward": float(np.std(rewards)),
        "min_reward": float(np.min(rewards)),
        "max_reward": float(np.max(rewards)),
        "num_episodes": len(rewards),
        "per_difficulty": {},
    }
    for info, r in zip(episode_infos, episode_rewards):
        d = info.get("difficulty", "unknown")
        out["per_difficulty"].setdefault(d, []).append(r)
    out["per_difficulty"] = {
        k: {"mean": float(np.mean(v)), "count": len(v)}
        for k, v in out["per_difficulty"].items()
    }
    return out


def print_summary(metrics):
    table = Table(title="Evaluation Summary", show_lines=True)
    table.add_column("Metric", style="bold")
    table.add_column("Value", justify="right")
    table.add_row("Episodes", str(metrics["num_episodes"]))
    table.add_row("Mean Reward", f"{metrics['mean_reward']:+.4f}")
    table.add_row("Std Reward", f"{metrics['std_reward']:.4f}")
    table.add_row("Min Reward", f"{metrics['min_reward']:+.4f}")
    table.add_row("Max Reward", f"{metrics['max_reward']:+.4f}")
    console.print(table)

    if metrics["per_difficulty"]:
        diff_table = Table(title="By Difficulty", show_lines=True)
        diff_table.add_column("Difficulty", style="bold")
        diff_table.add_column("Count", justify="right")
        diff_table.add_column("Mean Reward", justify="right")
        for diff, vals in metrics["per_difficulty"].items():
            color = "green" if vals["mean"] > 0 else "red"
            diff_table.add_row(diff, str(vals["count"]), f"[{color}]{vals['mean']:+.4f}[/]")
        console.print(diff_table)

## Evaluate agent across all scenarios

In [3]:
def evaluate_agent(agent, difficulty=None, verbose=True):
    env = MemoryEnv(difficulty=difficulty)
    num_scenarios = len(env._env.scenarios)

    all_rewards = []
    all_infos = []

    for ep in range(num_scenarios):
        total_reward, final_info = run_scenario(agent, env=env, verbose=verbose)
        all_rewards.append(total_reward)
        all_infos.append({
            "scenario_id": final_info.get("scenario_id", ""),
            "difficulty": final_info.get("difficulty", ""),
        })
        if verbose:
            color = "green" if total_reward > 0 else "red"
            console.print(f"  Scenario {final_info.get('scenario_id','?')} → "
                          f"[bold {color}]{total_reward:+.4f}[/]")

    return compute_metrics(all_rewards, all_infos)

## Run baseline evaluation

In [4]:
console.print("[bold magenta]=== Baseline Agent Evaluation ===[/]\n")
baseline = BaselineAgent()
metrics = evaluate_agent(baseline, difficulty=None, verbose=True)
print_summary(metrics)

=== Baseline Agent Evaluation ===

  Step  1 | Phase: query | Action: store_fact           | Reward: +0.0750
  Step  2 | Phase: query | Action: retrieve_memory      | Reward: +0.4700


Scenario easy_01 → +0.5450

  Step  1 | Phase: query | Action: store_fact           | Reward: +0.0750
  Step  2 | Phase: query | Action: retrieve_memory      | Reward: +0.4700


Scenario easy_02 → +0.5450

...

Scenario hard_02 → -0.0050

  Step  1 | Phase: store | Action: store_emotion        | Reward: +0.0750
  Step  2 | Phase: store | Action: store_preference     | Reward: +0.0750
  Step  3 | Phase: query | Action: store_fact           | Reward: +0.0750
  Step  4 | Phase: query | Action: retrieve_memory      | Reward: -0.1600


Scenario hard_03 → +0.0650

   Evaluation Summary    
┏━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Metric      ┃   Value ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━┩
│ Episodes    │       9 │
├─────────────┼─────────┤
│ Mean Reward │ +0.2075 │
├─────────────┼─────────┤
│ Std Reward  │  0.2824 │
├─────────────┼─────────┤
│ Min Reward  │ -0.1550 │
├─────────────┼─────────┤
│ Max Reward  │ +0.5450 │
└─────────────┴─────────┘

           By Difficulty            
┏━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━┓
┃ Difficulty ┃ Count ┃ Mean Reward ┃
┡━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━┩
│ easy       │     3 │     +0.5450 │
├────────────┼───────┼─────────────┤
│ medium     │     3 │     +0.0608 │
├────────────┼───────┼─────────────┤
│ hard       │     3 │     +0.0167 │
└────────────┴───────┴─────────────┘

## (Optional) Run RL agent evaluation

Requires a trained model at `models/ppo_memory_agent`.

In [ ]:
# from pathlib import Path
# rl_agent = RLAgent(model_path=Path("../models/ppo_memory_agent"))
# console.print("[bold magenta]=== RL Agent Evaluation ===[/]\n")
# rl_metrics = evaluate_agent(rl_agent, difficulty=None, verbose=True)
# print_summary(rl_metrics)